# Analyse Tokens: Optimize Run Review

This notebook inspects tokenizer optimize artifacts across systems, summarizes winner parameters, and visualizes candidate search behavior from `optimize_summary.json` and `optimize_runs.parquet`.

For TF-IDF artifact distribution and bucket-boundary analysis, use `notebooks/analyse_tfidf.ipynb`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import polars as pl
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import _bootstrap  # noqa: F401
from company_tokenize import resolve_active_optimize_sweep_dir
from company_tokenize.training import (
    OPTIMIZE_RUN_LOG_FILENAME,
    OPTIMIZE_SUMMARY_FILENAME,
)

from workspace.artifact_layout import tokenizer_artifact_root
from workspace.roots import default_workspace_roots

ROOTS = default_workspace_roots(ROOT)

# Leave empty to auto-discover systems with an optimize run on disk.
SYSTEMS = []
EXCLUDED_SYSTEMS = {"bert", "global"}
# The tokenizer whose optimize run is reviewed: an optimize run sits in its
# tokenizer's own folder, so one is read per tokenizer and encoding.
TOKENIZER = "wordpiece"  # wordpiece | sentencepiece
TOKENIZER_ENCODING = None  # bpe | unigram, for sentencepiece only
# Every scope's working directory sits beneath this one, named for its system.
SCOPES_ROOT = tokenizer_artifact_root(ROOTS)


def optimize_run_files(system: str) -> tuple[Path, Path] | None:
    """The summary and run log of a system's most recent optimize run for
    TOKENIZER, or None when there is none. The package resolves the folder,
    which is keyed by the corpus and the grid the run explored."""
    run_dir = resolve_active_optimize_sweep_dir(
        tokenizer_root=SCOPES_ROOT,
        scope="country",
        system=system,
        profile="default",
        trainer=TOKENIZER,
        tokenizer_encoding=TOKENIZER_ENCODING,
    )
    if run_dir is None:
        return None
    summary_path = run_dir / OPTIMIZE_SUMMARY_FILENAME
    run_log_path = run_dir / OPTIMIZE_RUN_LOG_FILENAME
    if not (summary_path.exists() and run_log_path.exists()):
        return None
    return summary_path, run_log_path


def resolve_optimize_systems() -> list[str]:
    if SYSTEMS:
        return [s for s in SYSTEMS if s not in EXCLUDED_SYSTEMS]
    if not SCOPES_ROOT.exists():
        raise FileNotFoundError(f"No tokenizer scope directory under: {SCOPES_ROOT}")

    return [
        path.name
        for path in sorted(SCOPES_ROOT.iterdir())
        if path.is_dir()
        and path.name not in EXCLUDED_SYSTEMS
        and optimize_run_files(path.name) is not None
    ]


SYSTEMS = resolve_optimize_systems()
if not SYSTEMS:
    print(
        f"No {TOKENIZER} optimize run found in any scope under {SCOPES_ROOT}. Make one "
        "with: uv run python scripts/train_tokenizer.py --mode optimize "
        "--systems <code>"
    )
else:
    print("Optimize analysis systems:", ", ".join(SYSTEMS))

display(pd.DataFrame({"system": SYSTEMS}))

In [ ]:
def load_system_artifacts(system: str) -> dict[str, object]:
    files = optimize_run_files(system)
    if files is None:
        raise FileNotFoundError(f"No {TOKENIZER} optimize run on disk for {system}")
    summary_path, run_log_path = files

    with summary_path.open("r", encoding="utf-8") as handle:
        summary = json.load(handle)

    run_df_local = pl.read_parquet(run_log_path).to_pandas()
    run_df_local["system"] = system

    return {
        "system": system,
        "summary_path": summary_path,
        "run_log_path": run_log_path,
        "summary": summary,
        "candidate_stats": summary.get("candidate_stats", {}),
        "winner_candidate": summary.get("winner_candidate", {}),
        "winner_output": summary.get("winner_output", {}),
        "run_df": run_df_local,
    }


if not SYSTEMS:
    system_records = []
    run_df = pd.DataFrame()
    comparison_df = pd.DataFrame()
    print("Skipping optimize artifact load because no systems were discovered.")
else:
    system_records = [load_system_artifacts(system) for system in SYSTEMS]
    run_df = pd.concat(
        [record["run_df"] for record in system_records], ignore_index=True
    )

    comparison_rows = []
    for record in system_records:
        summary = record["summary"]
        candidate_stats = record["candidate_stats"]
        winner_candidate = record["winner_candidate"]
        winner_output = record["winner_output"]
        winner_metrics = winner_output.get("post_retrain_metrics", {})
        comparison_rows.append(
            {
                "system": record["system"],
                "session_id": summary.get("session_id"),
                "total_candidates": candidate_stats.get("total_candidates"),
                "accepted_candidates": candidate_stats.get("accepted_candidates"),
                "rejected_candidates": candidate_stats.get("rejected_candidates"),
                "winner_vocab_requested": winner_candidate.get("vocab_size_requested"),
                "winner_vocab_resolved": winner_candidate.get("vocab_size_resolved"),
                "winner_min_frequency": winner_candidate.get("min_frequency"),
                "winner_median_fertility_distance": winner_candidate.get(
                    "median_fertility_distance"
                ),
                "winner_pass_rate": winner_candidate.get("pass_rate"),
                "winner_fertility": winner_metrics.get("fertility"),
                "winner_single_char_token_pct": winner_metrics.get(
                    "single_char_token_pct"
                ),
                "winner_token_count_median": winner_metrics.get("token_count_median"),
                "winner_token_count_p95": winner_metrics.get("token_count_p95"),
                "winner_unk_rate": winner_metrics.get("unk_rate"),
                "systems_in_run": ",".join(summary.get("systems", [])),
            }
        )

    comparison_df = pd.DataFrame(comparison_rows).sort_values("system")
    display(comparison_df)

print("Optimize rows by system:")
display(
    pd.DataFrame(
        {
            "system": SYSTEMS,
            "optimize_rows": [len(record["run_df"]) for record in system_records],
        }
    )
    if system_records
    else pd.DataFrame(columns=["system", "optimize_rows"])
)

No optimize artifacts found for discovered systems. Skip optimize-only sections.
Optimize rows by system (if available):


,system,optimize_rows


## System Summary

The first table compares the winning values for each discovered system.

Additional values are computed in the notebook only and shown separately.

Selection rationale still uses the vocab-size tolerance policy: among eligible candidates, pick the smallest effective vocab size within the configured fertility-distance band of the best candidate.

In [ ]:
if comparison_df.empty:
    print("No optimize comparison data available.")
else:
    display(comparison_df)

    selection_tolerance = (
        comparison_df.iloc[0].get("winner_median_fertility_distance")
        if not comparison_df.empty
        else None
    )
    derived_winner_rows = [
        {"field": "systems", "value": ", ".join(SYSTEMS)},
        {"field": "systems_with_runs", "value": len(comparison_df)},
        {
            "field": "total_candidates",
            "value": comparison_df["total_candidates"].sum()
            if not comparison_df.empty
            else None,
        },
        {
            "field": "accepted_candidates",
            "value": comparison_df["accepted_candidates"].sum()
            if not comparison_df.empty
            else None,
        },
        {
            "field": "rejected_candidates",
            "value": comparison_df["rejected_candidates"].sum()
            if not comparison_df.empty
            else None,
        },
        {
            "field": "winner_vocab_requested (by system)",
            "value": ", ".join(
                str(v) for v in comparison_df["winner_vocab_requested"].tolist()
            )
            if "winner_vocab_requested" in comparison_df
            else None,
        },
        {
            "field": "winner_vocab_resolved (by system)",
            "value": ", ".join(
                str(v) for v in comparison_df["winner_vocab_resolved"].tolist()
            )
            if "winner_vocab_resolved" in comparison_df
            else None,
        },
        {
            "field": "winner_min_frequency (by system)",
            "value": ", ".join(
                str(v) for v in comparison_df["winner_min_frequency"].tolist()
            )
            if "winner_min_frequency" in comparison_df
            else None,
        },
        {
            "field": "winner_median_fertility_distance (by system)",
            "value": ", ".join(
                str(v)
                for v in comparison_df["winner_median_fertility_distance"].tolist()
            )
            if "winner_median_fertility_distance" in comparison_df
            else None,
        },
        {
            "field": "winner_fertility (by system)",
            "value": ", ".join(
                str(v) for v in comparison_df["winner_fertility"].tolist()
            )
            if "winner_fertility" in comparison_df
            else None,
        },
        {
            "field": "winner_single_char_token_pct (by system)",
            "value": ", ".join(
                str(v) for v in comparison_df["winner_single_char_token_pct"].tolist()
            )
            if "winner_single_char_token_pct" in comparison_df
            else None,
        },
        {
            "field": "winner_unk_rate (by system)",
            "value": ", ".join(
                str(v) for v in comparison_df["winner_unk_rate"].tolist()
            )
            if "winner_unk_rate" in comparison_df
            else None,
        },
    ]
    display(pd.DataFrame(derived_winner_rows))
    display(
        comparison_df[
            [
                "system",
                "session_id",
                "winner_vocab_requested",
                "winner_vocab_resolved",
                "winner_fertility",
                "winner_single_char_token_pct",
                "winner_unk_rate",
            ]
        ]
    )

Optimize comparison data is empty. Run Cell 3 after generating optimize artifacts.


## Fertility and Single-Character Token Usage

This view focuses on how the candidate pool behaved on the validation split with fertility and single-character token usage as the primary diagnostics.

Because UNK is consistently zero in these runs, UNK-specific charts are intentionally omitted to keep the analysis focused on higher-signal differences across vocab and min-frequency choices.

How to read the box plot:

- Middle line: median metric value for that group.
- Box: middle 50% of runs (25th to 75th percentile).
- Whiskers: typical spread beyond the box (up to 1.5x IQR).
- Dots: outliers beyond whiskers.

The plots below use effective vocab size (resolved when available), ordered numerically, so families are compared in the intended scale order.

In [ ]:
if run_df.empty:
    print(
        "No optimize run data loaded. Skip this section unless optimize artifacts are available."
    )
else:
    run_df = run_df.copy()
    run_df["vocab_size_effective"] = run_df["vocab_size_resolved"].fillna(
        run_df["vocab_size_requested"]
    )
    run_df["vocab_size_effective"] = pd.to_numeric(
        run_df["vocab_size_effective"], errors="coerce"
    )
    run_df["vocab_label"] = run_df["vocab_size_effective"].map(
        lambda v: f"{int(v)}" if pd.notna(v) else "unknown"
    )
    run_df["min_frequency_label"] = run_df["min_frequency"].astype(str)

    ordered_systems = [
        system for system in SYSTEMS if system in run_df["system"].unique()
    ]
    ordered_vocab_values = sorted(
        v for v in run_df["vocab_size_effective"].dropna().unique()
    )
    ordered_vocab_labels = [str(int(v)) for v in ordered_vocab_values]
    if run_df["vocab_label"].eq("unknown").any():
        ordered_vocab_labels.append("unknown")

    grouped = (
        run_df.groupby(["system", "vocab_label", "min_frequency_label"], dropna=False)
        .agg(
            total_runs=("rejected", "size"),
            accepted_runs=("rejected", lambda s: (~s).sum()),
            pass_rate=("rejected", lambda s: (~s).mean()),
            median_single_char_token_pct=("single_char_token_pct", "median"),
            median_fertility=("fertility", "median"),
            mean_fertility=("fertility", "mean"),
            mean_single_char_token_pct=("single_char_token_pct", "mean"),
        )
        .reset_index()
        .sort_values(["system", "vocab_label", "min_frequency_label"])
    )

    display(grouped)

    single_char_by_vocab = (
        run_df.groupby(["system", "vocab_label"], dropna=False)
        .agg(
            runs=("single_char_token_pct", "size"),
            median_single_char_token_pct=("single_char_token_pct", "median"),
            mean_single_char_token_pct=("single_char_token_pct", "mean"),
            accepted_runs=("rejected", lambda s: (~s).sum()),
        )
        .reset_index()
        .sort_values(["system", "vocab_label"])
    )
    single_char_by_vocab["accept_rate"] = (
        single_char_by_vocab["accepted_runs"] / single_char_by_vocab["runs"]
    )
    display(single_char_by_vocab)

    fig_single_char_vocab = px.bar(
        single_char_by_vocab,
        x="vocab_label",
        y="median_single_char_token_pct",
        color="system",
        barmode="group",
        hover_data=[
            "runs",
            "accepted_runs",
            "accept_rate",
            "mean_single_char_token_pct",
        ],
        category_orders={
            "system": ordered_systems,
            "vocab_label": ordered_vocab_labels,
        },
        title="Single-character token usage (median %) by effective vocab size and system",
        labels={
            "vocab_label": "Effective vocab size",
            "median_single_char_token_pct": "Median single-character token usage (%)",
        },
    )
    fig_single_char_vocab.show()

    fig1 = px.box(
        run_df,
        x="vocab_label",
        y="fertility",
        color="rejected",
        facet_col="system",
        points="outliers",
        category_orders={
            "system": ordered_systems,
            "vocab_label": ordered_vocab_labels,
            "rejected": [False, True],
        },
        title="Validation fertility grouped by effective vocab size and system",
        labels={
            "vocab_label": "Effective vocab size",
            "fertility": "Fertility",
            "rejected": "Rejected",
        },
    )
    fig1.show()

    fig2 = px.box(
        run_df,
        x="vocab_label",
        y="single_char_token_pct",
        color="rejected",
        facet_col="system",
        points="outliers",
        category_orders={
            "system": ordered_systems,
            "vocab_label": ordered_vocab_labels,
            "rejected": [False, True],
        },
        title="Validation single-character token usage (%) grouped by effective vocab size and system",
        labels={
            "vocab_label": "Effective vocab size",
            "single_char_token_pct": "Single-character token usage (%)",
            "rejected": "Rejected",
        },
    )
    fig2.show()

    fig3 = px.line(
        grouped.sort_values(["system", "min_frequency_label", "vocab_label"]),
        x="vocab_label",
        y="median_fertility",
        color="min_frequency_label",
        facet_col="system",
        markers=True,
        category_orders={
            "system": ordered_systems,
            "vocab_label": ordered_vocab_labels,
        },
        hover_data=[
            "accepted_runs",
            "total_runs",
            "pass_rate",
            "median_single_char_token_pct",
        ],
        title="Median fertility by effective vocab size, min-frequency, and system",
        labels={
            "vocab_label": "Effective vocab size",
            "median_fertility": "Median fertility",
            "min_frequency_label": "Min frequency",
        },
    )
    fig3.show()

    for system in ordered_systems:
        fertility_heat = grouped[grouped["system"] == system].pivot_table(
            index="min_frequency_label",
            columns="vocab_label",
            values="median_fertility",
            aggfunc="first",
        )
        fertility_heat = fertility_heat.reindex(
            index=sorted(fertility_heat.index, key=lambda x: int(x))
        )
        fertility_heat = fertility_heat.reindex(
            columns=[
                label
                for label in ordered_vocab_labels
                if label in fertility_heat.columns
            ]
        )

        fig4 = px.imshow(
            fertility_heat,
            aspect="auto",
            title=f"Median fertility heatmap by min-frequency and effective vocab size: {system}",
            labels={
                "x": "Effective vocab size",
                "y": "Min frequency",
                "color": "Median fertility",
            },
        )
        fig4.show()

    fertility_df = run_df.dropna(subset=["fertility"]).copy()
    fertility_df["fertility_bin"] = pd.cut(
        fertility_df["fertility"], bins=10, include_lowest=True
    )
    fertility_summary = (
        fertility_df.groupby(["system", "fertility_bin"], observed=False)
        .agg(
            total_runs=("rejected", "size"),
            accepted_runs=("rejected", lambda s: (~s).sum()),
        )
        .reset_index()
    )
    fertility_summary["accept_rate"] = (
        fertility_summary["accepted_runs"] / fertility_summary["total_runs"]
    )
    fertility_summary["fertility_bin"] = fertility_summary["fertility_bin"].astype(str)

    fig5 = px.bar(
        fertility_summary,
        x="fertility_bin",
        y="accept_rate",
        color="system",
        barmode="group",
        hover_data=["total_runs", "accepted_runs"],
        title="Acceptance rate by fertility bin and system",
        labels={"fertility_bin": "Fertility bin", "accept_rate": "Acceptance rate"},
    )
    fig5.update_layout(xaxis_tickangle=-35)
    fig5.show()

    low_fertility_cutoff = fertility_df["fertility"].quantile(0.25)
    low_fertility_rejects = fertility_df[
        (fertility_df["rejected"]) & (fertility_df["fertility"] <= low_fertility_cutoff)
    ].copy()
    if low_fertility_rejects.empty:
        print("No rejected candidates in the lowest fertility quartile.")
    else:
        reason_counts = (
            low_fertility_rejects.groupby(["system", "rejection_reason"], dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values(["system", "count"], ascending=[True, False])
        )
        display(reason_counts)

No optimize run data loaded. Skip this section unless optimize artifacts are available.


## Accepted vs Rejected Samples

Review a few candidates at the extremes to understand why the optimizer rejected them.

In [5]:
if run_df.empty:
    print("No optimize run data loaded. Skip accepted/rejected sample inspection.")
else:
    accepted = run_df.loc[~run_df["rejected"]].copy()
    rejected = run_df.loc[run_df["rejected"]].copy()

    print("Top accepted by lowest absolute fertility:")
    display(
        accepted.sort_values(["system", "fertility", "elapsed_seconds"]).head(10)[
            [
                "system",
                "seed",
                "vocab_size_requested",
                "min_frequency",
                "unk_rate",
                "single_char_token_pct",
                "fertility",
                "fertility_distance",
                "elapsed_seconds",
            ]
        ]
    )

    print("Top rejected by highest absolute fertility:")
    display(
        rejected.sort_values(
            ["system", "fertility", "elapsed_seconds"], ascending=[True, False, True]
        ).head(10)[
            [
                "system",
                "seed",
                "vocab_size_requested",
                "min_frequency",
                "rejection_reason",
                "unk_rate",
                "single_char_token_pct",
                "fertility",
                "fertility_distance",
                "elapsed_seconds",
            ]
        ]
    )

No optimize run data loaded. Skip accepted/rejected sample inspection.
